# Cleaning Dataset Tanaman Pangan per Kecamatan — Kabupaten Kediri

Membersihkan satu file: `Tanaman_Pangan_per_Kecamatan.csv` (130 baris = 26 kecamatan × 5 komoditas).

**Hasil pengecekan isi file (sebelum menulis kode):**
- Kolom: `kecamatan, tanaman, luas_panen_ha, produksi_ton, produktivitas_kuintal_ha`. Sudah *long format* dan satuan sudah ada di nama kolom.
- Tidak ada duplikat, tidak ada nama kecamatan di luar 26 master, tiap komoditas punya tepat 26 baris.
- `produktivitas_kuintal_ha` kosong di 7 baris. Semuanya baris yang luas panennya 0, jadi memang tidak bisa dihitung (bukan data hilang).
- Dua baris (Kedelai di Gampengrejo dan Pagu) berluas 0,00 ha tapi produksi 0,01 ton: artefak pembulatan 2 desimal.
- Total per komoditas cocok dengan total kabupaten dalam batas pembulatan.

**Yang dilakukan notebook ini:** memvalidasi struktur & kelengkapan, menyeragamkan nama kolom (`tanaman` → `komoditas`, sama seperti file hasil cleaning lain), mengecek konsistensi
produktivitas, membandingkan total dengan data kabupaten (kalau file hasil cleaning-nya ada), lalu menyimpan.

**Output:** `data/clean/tanaman_pangan/tanaman_pangan_per_kecamatan.csv`
→ kolom `kecamatan, komoditas, luas_panen_ha, produksi_ton, produktivitas_kuintal_ha`

In [ ]:
# 1. IMPORT LIBRARY
import os
import re

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", None)
print("✓ Library siap.")

In [ ]:
# 2. CARI LOKASI PROJECT (folder 'data') SECARA OTOMATIS
CURRENT_DIR = os.getcwd()
PROJECT_ROOT = None

search_dir = CURRENT_DIR
while True:
    if os.path.isdir(os.path.join(search_dir, "data")):
        PROJECT_ROOT = search_dir
        break
    parent_dir = os.path.dirname(search_dir)
    if parent_dir == search_dir:
        break
    search_dir = parent_dir

if PROJECT_ROOT is None:
    raise FileNotFoundError(
        "Folder 'data' tidak ditemukan dari direktori notebook ini ke atas. "
        "Pastikan notebook dijalankan dari dalam repository (mis. dari folder 'notebooks/')."
    )

DATA_DIR = os.path.join(PROJECT_ROOT, "data")
OUT_DIR = os.path.join(DATA_DIR, "clean", "tanaman_pangan")
os.makedirs(OUT_DIR, exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("Output      :", OUT_DIR)

In [ ]:
# 3. CARI FILE SUMBER
# Dicari di seluruh data/ KECUALI folder hasil (clean, processed), supaya file keluaran tidak terbaca
# sebagai sumber. Nama dicocokkan longgar: harus memuat 'tanaman', 'pangan', 'kecamatan'.

FOLDER_HASIL = {"clean", "processed"}

def cari_file_sumber():
    kandidat = []
    for root, dirs, files in os.walk(DATA_DIR):
        dirs[:] = [d for d in dirs if d.lower() not in FOLDER_HASIL]
        for f in files:
            nama = f.lower()
            if nama.endswith(".csv") and all(k in nama for k in ("tanaman", "pangan", "kecamatan")):
                kandidat.append(os.path.join(root, f))
    kandidat.sort(key=lambda p: (os.path.basename(os.path.dirname(p)) != "tanaman_pangan", p))
    return kandidat

kandidat = cari_file_sumber()
if not kandidat:
    raise FileNotFoundError(f"File tanaman pangan per kecamatan tidak ditemukan di {DATA_DIR}")
if len(kandidat) > 1:
    print("⚠️  Lebih dari satu kandidat, dipakai yang pertama:")
    for k in kandidat:
        print("    ", k)

FILE_SUMBER = kandidat[0]
print("File sumber:", FILE_SUMBER)

In [ ]:
# 4. MASTER KECAMATAN & FUNGSI BANTU

MASTER_KECAMATAN = [
    "Mojo", "Semen", "Ngadiluwih", "Kras", "Ringinrejo", "Kandat", "Wates", "Ngancar",
    "Plosoklaten", "Gurah", "Puncu", "Kepung", "Kandangan", "Pare", "Badas", "Kunjang",
    "Plemahan", "Purwoasri", "Papar", "Pagu", "Kayenkidul", "Gampengrejo", "Ngasem",
    "Banyakan", "Grogol", "Tarokan",
]
KOREKSI_KECAMATAN = {"Kayen Kidul": "Kayenkidul"}


def ke_angka(teks):
    # '-' -> 0.0 (konvensi BPS: tidak ada). Kosong / '...' -> NaN (tidak tersedia / tidak dapat dihitung).
    # Mengenali format Indonesia (1.234,56) dan internasional (1,234.56).
    if pd.isna(teks):
        return np.nan
    s = str(teks).strip()
    if s == "-":
        return 0.0
    if s in ("", "...", "nan"):
        return np.nan
    s = re.sub(r"[^0-9.,-]", "", s)
    if "," in s and "." in s:
        if s.rfind(",") > s.rfind("."):
            s = s.replace(".", "").replace(",", ".")
        else:
            s = s.replace(",", "")
    elif "," in s:
        if re.fullmatch(r"-?[1-9]\d{0,2}(,\d{3})+", s):
            s = s.replace(",", "")
        else:
            s = s.replace(",", ".")
    try:
        return float(s)
    except ValueError:
        return np.nan


def rapikan_nama(teks):
    return re.sub(r"\s+", " ", str(teks)).strip()


print("✓ Master kecamatan & fungsi bantu siap.")

In [ ]:
# 5. BACA FILE

# dtype=str + keep_default_na=False: sel kosong tetap kosong (bukan diubah pandas diam-diam).
df_raw = pd.read_csv(FILE_SUMBER, encoding="utf-8-sig", dtype=str, keep_default_na=False)

KOLOM_WAJIB = ["kecamatan", "tanaman", "luas_panen_ha", "produksi_ton", "produktivitas_kuintal_ha"]
kurang = [k for k in KOLOM_WAJIB if k not in df_raw.columns]
if kurang:
    raise ValueError(f"Kolom tidak ditemukan: {kurang}. Kolom yang ada: {list(df_raw.columns)}")

# Buang baris yang seluruh selnya kosong (mis. baris kosong di akhir file).
df_raw = df_raw[~df_raw[KOLOM_WAJIB].apply(lambda r: all(str(x).strip() == "" for x in r), axis=1)]
df_raw = df_raw.reset_index(drop=True)

print(f"{len(df_raw)} baris terbaca.")
display(df_raw.head())

In [ ]:
# 6. BERSIHKAN

df = pd.DataFrame({
    "kecamatan": df_raw["kecamatan"].apply(rapikan_nama).replace(KOREKSI_KECAMATAN),
    "komoditas": df_raw["tanaman"].apply(rapikan_nama),
    "luas_panen_ha": df_raw["luas_panen_ha"].apply(ke_angka),
    "produksi_ton": df_raw["produksi_ton"].apply(ke_angka),
    "produktivitas_kuintal_ha": df_raw["produktivitas_kuintal_ha"].apply(ke_angka),
})

# Nilai teks yang ada isinya tapi gagal dibaca jadi angka -> laporkan (bukan diam-diam jadi NaN).
for kolom in ["luas_panen_ha", "produksi_ton", "produktivitas_kuintal_ha"]:
    ada_isi = df_raw[kolom].str.strip() != ""
    gagal = ada_isi & df[kolom].isna()
    if gagal.any():
        print(f"⚠️  {gagal.sum()} nilai '{kolom}' tidak terbaca sebagai angka:")
        display(df_raw.loc[gagal, ["kecamatan", "tanaman", kolom]])

df = df.sort_values(["komoditas", "kecamatan"]).reset_index(drop=True)
display(df.head(10))

In [ ]:
# 7. VALIDASI STRUKTUR & KELENGKAPAN

# a) Kecamatan: harus tepat 26 dan sama dengan master
selisih_nama = set(df["kecamatan"]) ^ set(MASTER_KECAMATAN)
assert not selisih_nama, f"Nama kecamatan tidak sama dengan master: {selisih_nama}"

# b) Tidak ada duplikat, dan tiap komoditas lengkap 26 kecamatan
assert not df.duplicated(["kecamatan", "komoditas"]).any(), "Ada pasangan kecamatan-komoditas ganda."
jumlah_per_komoditas = df.groupby("komoditas")["kecamatan"].nunique()
tidak_lengkap = jumlah_per_komoditas[jumlah_per_komoditas != len(MASTER_KECAMATAN)]
assert tidak_lengkap.empty, f"Komoditas tidak lengkap 26 kecamatan:\n{tidak_lengkap}"

# c) Luas panen & produksi tidak boleh kosong atau negatif
assert df[["luas_panen_ha", "produksi_ton"]].notna().all().all(), "Ada luas panen/produksi yang kosong."
assert (df[["luas_panen_ha", "produksi_ton", "produktivitas_kuintal_ha"]].fillna(0) >= 0).all().all(), "Ada nilai negatif."

print(f"✓ {len(df)} baris = {df['kecamatan'].nunique()} kecamatan × {df['komoditas'].nunique()} komoditas, tanpa duplikat.")
print("  Komoditas:", sorted(df["komoditas"].unique()))

In [ ]:
# 8. VALIDASI PRODUKTIVITAS

# a) Produktivitas boleh kosong HANYA kalau luas panen = 0 (tidak bisa dihitung).
kosong_tapi_ada_luas = df[df["produktivitas_kuintal_ha"].isna() & (df["luas_panen_ha"] > 0)]
if len(kosong_tapi_ada_luas):
    print("⚠️  Produktivitas kosong padahal luas panen > 0:")
    display(kosong_tapi_ada_luas)
n_kosong = df["produktivitas_kuintal_ha"].isna().sum()
print(f"✓ {n_kosong} produktivitas kosong, semuanya pada baris luas panen = 0 (dibiarkan NaN, tidak diisi).")

# b) Konsistensi: produktivitas = produksi / luas panen x 10 (ton/ha -> kuintal/ha).
#    Toleransi memperhitungkan pembulatan 2 desimal pada ketiga kolom.
ada = df[df["luas_panen_ha"] > 0].copy()
hitung = ada["produksi_ton"] / ada["luas_panen_ha"] * 10
toleransi = 0.01 + 0.005 / ada["luas_panen_ha"] + 0.005 / ada["produksi_ton"].replace(0, np.nan)
selisih_rel = ((hitung - ada["produktivitas_kuintal_ha"]).abs() / hitung)
tidak_konsisten = ada[(selisih_rel > toleransi) & ada["produktivitas_kuintal_ha"].notna()]
if len(tidak_konsisten):
    print(f"⚠️  {len(tidak_konsisten)} baris produktivitas tidak konsisten dengan produksi/luas panen:")
    display(tidak_konsisten)
else:
    print("✓ Produktivitas konsisten dengan produksi / luas panen (dalam batas pembulatan).")

# c) Artefak pembulatan: luas 0,00 ha tapi produksi > 0
artefak = df[(df["luas_panen_ha"] == 0) & (df["produksi_ton"] > 0)]
if len(artefak):
    print("ℹ️  Luas panen 0,00 ha tetapi produksi > 0 (artefak pembulatan 2 desimal, nilai dibiarkan apa adanya):")
    display(artefak)

In [ ]:
# 9. CEK SILANG DENGAN DATA KABUPATEN (opsional)
# Memakai hasil cleaning tanaman_pangan_kabupaten.csv kalau sudah ada di folder output.

path_kab = os.path.join(OUT_DIR, "tanaman_pangan_kabupaten.csv")
if os.path.isfile(path_kab):
    kab = pd.read_csv(path_kab, encoding="utf-8-sig").set_index("komoditas")
    jumlah = df.groupby("komoditas")[["luas_panen_ha", "produksi_ton"]].sum()

    hanya_di_kec = set(jumlah.index) - set(kab.index)
    hanya_di_kab = set(kab.index) - set(jumlah.index)
    if hanya_di_kec or hanya_di_kab:
        print(f"⚠️  Komoditas tidak sama. Hanya di file kecamatan: {hanya_di_kec}; hanya di file kabupaten: {hanya_di_kab}")

    umum = sorted(set(jumlah.index) & set(kab.index))
    perbandingan = pd.DataFrame({
        "luas_kab": kab.loc[umum, "luas_panen_ha"],
        "luas_jumlah_kec": jumlah.loc[umum, "luas_panen_ha"].round(2),
        "produksi_kab": kab.loc[umum, "produksi_ton"].round(2),
        "produksi_jumlah_kec": jumlah.loc[umum, "produksi_ton"].round(2),
    })
    perbandingan["selisih_luas"] = (perbandingan["luas_jumlah_kec"] - perbandingan["luas_kab"]).round(2)
    perbandingan["selisih_produksi"] = (perbandingan["produksi_jumlah_kec"] - perbandingan["produksi_kab"]).round(2)
    display(perbandingan)

    # Tiap angka dibulatkan ke 0,01 sehingga galat maksimum jumlah 26 kecamatan = 26 x 0,005.
    BATAS_PEMBULATAN = 26 * 0.005 + 0.01
    lewat = perbandingan[(perbandingan["selisih_luas"].abs() > BATAS_PEMBULATAN)
                         | (perbandingan["selisih_produksi"].abs() > BATAS_PEMBULATAN)]
    if len(lewat):
        print(f"⚠️  Selisih melebihi batas pembulatan ({BATAS_PEMBULATAN:.2f}) untuk: {lewat.index.tolist()}")
    else:
        print(f"✓ Jumlah semua kecamatan sama dengan total kabupaten dalam batas pembulatan (±{BATAS_PEMBULATAN:.2f}).")
else:
    print("ℹ️  tanaman_pangan_kabupaten.csv belum ada di folder output, cek silang dilewati.")

In [ ]:
# 10. CEK KEWAJARAN UNTUK KEBUTUHAN SISTEM (peringatan, tidak mengubah data)

# a) Keseragaman produktivitas antar kecamatan.
#    Baris dengan luas < 1 ha diabaikan agar derau pembulatan tidak mengganggu.
CV_AMBANG = 0.02
LUAS_MIN_UNTUK_CV = 1.0
seragam, tak_terukur = [], []
for komoditas, g in df.groupby("komoditas"):
    g = g[g["luas_panen_ha"] >= LUAS_MIN_UNTUK_CV]
    if len(g) < 5:
        tak_terukur.append(komoditas)
        continue
    hasil = g["produksi_ton"] / g["luas_panen_ha"] * 10
    cv = hasil.std() / hasil.mean()
    if cv < CV_AMBANG:
        seragam.append((komoditas, cv))

if seragam:
    print(f"⚠️  Produktivitas hampir seragam antar kecamatan (variasi < {CV_AMBANG:.0%}):")
    for komoditas, cv in seragam:
        print(f"     {komoditas}: variasi {cv:.2%}")
    print("    Data pengamatan per kecamatan biasanya bervariasi jauh lebih besar (irigasi, tanah, praktik tani).")
    print("    Pola ini khas data hasil alokasi proporsional dari total kabupaten. Akibatnya clustering,")
    print("    prediksi, dan rekomendasi tingkat kecamatan hanya akan mencerminkan pola pembagiannya,")
    print("    bukan kondisi lapangan. Catat asal data ini di kamus data.")
if tak_terukur:
    print(f"ℹ️  Terlalu sedikit baris berluas >= {LUAS_MIN_UNTUK_CV} ha untuk menilai keseragaman: {tak_terukur}")

# b) Nilai terlalu kecil untuk bermakna
for komoditas, g in df.groupby("komoditas"):
    kecil = ((g["luas_panen_ha"] > 0) & (g["luas_panen_ha"] < 1)).sum()
    nol = (g["luas_panen_ha"] == 0).sum()
    if kecil + nol >= len(g) // 2:
        print(f"⚠️  {komoditas}: {kecil} kecamatan berluas < 1 ha dan {nol} berluas 0 dari {len(g)}. Terlalu kecil untuk analisis per kecamatan.")

# c) Komoditas di proposal yang belum ada (Ketela Pohon = Ubi Kayu)
SINONIM = {"Ketela Pohon": "Ubi Kayu"}
KOMODITAS_PROPOSAL = ["Padi", "Jagung", "Kedelai", "Kacang Tanah", "Kacang Hijau", "Ubi Kayu", "Ubi Jalar"]
ada_komoditas = {SINONIM.get(k, k) for k in df["komoditas"]}
belum = [k for k in KOMODITAS_PROPOSAL if k not in ada_komoditas]
if belum:
    print(f"⚠️  Komoditas di proposal yang belum ada di file ini: {belum}")

print("⚠️  File ini tidak punya kolom tahun. Catat tahun datanya dari sumber (lahan: 2023).")

In [ ]:
# 11. SIMPAN

path_out = os.path.join(OUT_DIR, "tanaman_pangan_per_kecamatan.csv")
df.to_csv(path_out, index=False, encoding="utf-8-sig")

# Baca ulang untuk memastikan isi file sama dengan di memori.
cek = pd.read_csv(path_out, encoding="utf-8-sig")
assert list(cek.columns) == ["kecamatan", "komoditas", "luas_panen_ha", "produksi_ton", "produktivitas_kuintal_ha"]
assert len(cek) == len(df)
assert np.allclose(cek[["luas_panen_ha", "produksi_ton"]], df[["luas_panen_ha", "produksi_ton"]])
assert cek["produktivitas_kuintal_ha"].isna().sum() == df["produktivitas_kuintal_ha"].isna().sum()

print("✓ Tersimpan di:", path_out)
display(cek)

## Yang perlu kamu perhatikan
1. **Sifat data.** Pola produktivitas yang hampir sama di semua kecamatan (lihat peringatan di cell 10) menandakan
   angka per kecamatan ini hasil pembagian dari total kabupaten, bukan pencatatan per kecamatan. File ini
   cocok untuk membangun alur ETL, skema database, dan tampilan awal dashboard. Untuk model tingkat kecamatan,
   ganti dengan data DKPP/BPS per kecamatan begitu tersedia. Sebaiknya asal data ini dicatat di kamus data.
2. **Kedelai** nilainya terlalu kecil per kecamatan (total kabupaten hanya 1,9 ha).
3. **Jagung dan Kacang Hijau belum ada.** "Ketela Pohon" dibiarkan sesuai sumber; penyeragaman ke "Ubi Kayu"
   sebaiknya lewat tabel master komoditas.
4. **Tidak ada tahun** di file ini.
5. Produktivitas yang kosong (7 baris) memang tidak bisa dihitung karena luas panennya 0; dibiarkan kosong.